# Roles, permissions, and least privilege  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.2.b: Detect least-privilege violations

**Skill:** Mitigate Risk · **Maps to:** 5.2.A, 5.2.B · **Language:** Python 3 (pytest)

## Problem
Build the audit function a security reviewer would run over a company's role assignments. Given a
map of roles to the permissions they grant, a map of users to their assigned roles, and a map of
users to the minimum permission set their job function actually requires, find every user who
holds at least one permission (via their assigned roles) that is NOT in their job function's
required set.

## Function contract
Implement `solve(roles: dict, user_roles: dict, job_requirements: dict) -> dict` in the code editor
(see the Reference section below).

## Input
- `roles`: `dict[str, list[str]]`, role name -> list of permissions it grants.
- `user_roles`: `dict[str, list[str]]`, user name -> list of role names assigned to them.
- `job_requirements`: `dict[str, list[str]]`, user name -> list of permissions their job function
  actually requires.

## Output
A dict mapping each violating user to the **sorted** list of their excess (unauthorized)
permissions. Users with no excess permissions are omitted entirely from the result.

## Examples
```
roles = {"support": ["read_ticket", "write_ticket_reply"], "billing": ["read_invoice", "issue_refund"]}
user_roles = {"dana": ["support", "billing"]}
job_requirements = {"dana": ["read_ticket", "write_ticket_reply"]}

solve(roles, user_roles, job_requirements)
-> {"dana": ["issue_refund", "read_invoice"]}
```

```
roles = {"support": ["read_ticket"]}
user_roles = {"sam": ["support"]}
job_requirements = {"sam": ["read_ticket"]}

solve(roles, user_roles, job_requirements)
-> {}
```

## Constraints
- Deterministic, offline, standard library only.
- A user with roles granting no permissions beyond their job requirements is omitted from the
  output entirely (not included with an empty list).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(roles: dict, user_roles: dict, job_requirements: dict) -> dict
```

### Input
- `roles`: `dict[str, list[str]]`, role name -> permissions it grants.
- `user_roles`: `dict[str, list[str]]`, user name -> assigned role names.
- `job_requirements`: `dict[str, list[str]]`, user name -> permissions their job requires. A user
  absent from this dict requires no permissions (empty set).

### Output (`result`)
`dict[str, list[str]]` mapping each user who has at least one excess permission to the **sorted**
list of their excess permissions (permissions granted by their roles but not in their job
requirements). Users with zero excess permissions are omitted entirely from the result.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Permissions from multiple assigned roles are unioned (deduplicated) before comparing against
  job requirements.
- Unknown role names in `user_roles` (not present in `roles`) contribute no permissions.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.2.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import Dict, List


def solve(
    roles: Dict[str, List[str]],
    user_roles: Dict[str, List[str]],
    job_requirements: Dict[str, List[str]],
) -> Dict[str, List[str]]:
    """Flag users whose role-derived permissions exceed their job requirements."""
    result: Dict[str, List[str]] = {}
    for user, assigned_roles in user_roles.items():
        granted = set()
        for role in assigned_roles:
            granted.update(roles.get(role, []))
        required = set(job_requirements.get(user, []))
        excess = granted - required
        if excess:
            result[user] = sorted(excess)
    return result


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_dana_has_excess():
    roles = {"support": ["read_ticket", "write_ticket_reply"], "billing": ["read_invoice", "issue_refund"]}
    user_roles = {"dana": ["support", "billing"]}
    job_requirements = {"dana": ["read_ticket", "write_ticket_reply"]}
    assert impl.solve(roles, user_roles, job_requirements) == {"dana": ["issue_refund", "read_invoice"]}


def test_sam_no_excess_omitted():
    roles = {"support": ["read_ticket"]}
    user_roles = {"sam": ["support"]}
    job_requirements = {"sam": ["read_ticket"]}
    assert impl.solve(roles, user_roles, job_requirements) == {}


def test_multiple_users_mixed():
    roles = {"support": ["read_ticket", "write_ticket_reply"], "admin": ["delete_user", "read_ticket"]}
    user_roles = {"a": ["support"], "b": ["admin"]}
    job_requirements = {"a": ["read_ticket", "write_ticket_reply"], "b": ["read_ticket"]}
    assert impl.solve(roles, user_roles, job_requirements) == {"b": ["delete_user"]}


# ---- hidden / edge cases -------------------------------------------------
def test_user_with_no_roles():
    roles = {"support": ["read_ticket"]}
    user_roles = {"ghost": []}
    job_requirements = {"ghost": []}
    assert impl.solve(roles, user_roles, job_requirements) == {}


def test_user_missing_from_job_requirements_treated_as_none_required():
    roles = {"support": ["read_ticket"]}
    user_roles = {"newhire": ["support"]}
    job_requirements = {}
    assert impl.solve(roles, user_roles, job_requirements) == {"newhire": ["read_ticket"]}


def test_overlapping_roles_deduplicated():
    roles = {"r1": ["p1", "p2"], "r2": ["p2", "p3"]}
    user_roles = {"u": ["r1", "r2"]}
    job_requirements = {"u": ["p1", "p2", "p3"]}
    assert impl.solve(roles, user_roles, job_requirements) == {}


def test_empty_inputs():
    assert impl.solve({}, {}, {}) == {}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
